In [11]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler

# ---- Load data ----
df = pd.read_csv("/Users/kimballweeks/Downloads/cleaned_data_v3.csv")

# ---- Rename to simpler consistent names ----
df = df.rename(columns={
    "pct_highschool_or_more (1990)": "edu_1990",
    "income_1989_real_2023": "inc_1989",
    "Pop 2023": "pop_2023",
    "Pop 2010": "pop_2010",
    "Established firms 2022": "firms_2022",
    "Established firms 1989": "firms_1989",
})

# ---- Drop rows with missing or zero population ----
df = df.dropna(subset=["Index_v2","inc_1989","edu_1990","pop_2010","firms_2022"])
df = df[df["pop_2010"] > 0]

# ---- Create rate & asinh DV ----
df["rate_2022_per_1k"] = 1000 * df["firms_2022"] / df["pop_2010"]   # or pop_2022 if you have it
df["dv_asinh"] = np.arcsinh(df["rate_2022_per_1k"])                # handles zeros automatically

# ---- Standardize income & education ----
scaler = StandardScaler()
df[["inc_1989_scaled","edu_1990_scaled"]] = scaler.fit_transform(df[["inc_1989","edu_1990"]])

# ---- Preferred regression ----
model = smf.ols(
    "dv_asinh ~ Index_v2 + inc_1989_scaled + edu_1990_scaled + np.log(pop_2010) + C(State)",
    data=df
).fit(cov_type="cluster", cov_kwds={"groups": df["State"]})

print(model.summary())


TypeError: unsupported operand type(s) for /: 'str' and 'int'